# **Fine-Tuning Phi-3-Mini for Wellness Coaching**

This notebook demonstrates the fine-tuning of the microsoft/phi-3-mini-4k-instruct model to transform it from a general assistant into a warm, empathetic Wellness Coach.

Key Improvements:

- Transition from clinical/robotic lists to human-centric validation.

- Implementation of safety guardrails for high-stress scenarios (e.g., sleep deprivation).

- Quantized training (QLoRA) for efficiency.

In [1]:
!pip install transformers peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00


In [3]:
from google.colab import files
uploaded = files.upload()

Saving phi3-wellness-lora-v2.zip to phi3-wellness-lora-v2.zip


In [4]:
import zipfile

with zipfile.ZipFile("phi3-wellness-lora-v2.zip", 'r') as zip_ref:
    zip_ref.extractall("phi3-wellness-lora-v2")

**1. Model Loading with 4-bit Quantization**

We use BitsAndBytesConfig to load the model in 4-bit. This allows us to run inference and training on consumer-grade GPUs like the Tesla T4 while maintaining high performance.

In [5]:
from transformers import BitsAndBytesConfig
import torch

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "microsoft/phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [9]:
from peft import PeftModel

model = PeftModel.from_pretrained(base_model, "phi3-wellness-lora-v2")

**2. Optimized Inference Parameters**

To ensure the coach sounds natural and doesn't cut off mid-sentence, we use the following settings:

- Temperature (0.3): Keeps the model focused and grounded.

- Max New Tokens (750): Ensures complex wellness advice is fully completed.

- Repetition Penalty (1.2): Prevents the model from getting stuck in "empathy loops."

In [23]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

def generate_response(prompt):
    input_text = f"""<|user|>
    \n{prompt}\n\n<|assistant|>\n
    """

    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=750,
        do_sample=True,
        temperature=0.3,        # 🔥 LOWER THIS
        top_p=0.85,             # 🔥 tighter sampling
        repetition_penalty=1.2, # 🔥 reduce generic loops
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [40]:
print(generate_response("I missed my heart medication today because I was busy. What should I do?"))

I missed my heart medication today because I was busy. What should I do?

 It happens! Life gets hectic, and sometimes we skip things—but usually our bodies are more forgiving than the clock is strict with us. 🕒 Don’t sweat it too much; just take your next one on time when you can remember to put that pill bottle back in its spot (maybe even set an alarm if needed). The most important thing isn't a perfect streak of taking pills at exact times —it's keeping them going consistently enough so they keep doing their job for as long as possible. If there’s any way to check-in about how everything feels or get specific advice from someone who knows exactly what those numbers mean together (*like* calling your doc), go ahead now while this small hiccup hasn’t had time to turn into something bigger yet.* Your health team wants only two outcomes: safety AND stability above all else right here.” So let me reassure myself again before moving forward…if missing once really throws off anything majo